# Feature 138: alpha tuning on the biased dataset

This notebook replays the original open-prompt dataset (`a picture of a {race} {gender}`) and applies the feature 138 alpha sweep at the same SAE block. It does not use the prompt-engineered neutral set or hand-authored headscarf/no-headscarf prompts. The saved generated images are originals from `data/datasets/biased/images`; the manifest supplies each exact prompt and seed.

The original focused run uses `middle_eastern_woman`, the group where feature 138 peaked in the biased-set disparity report. The all-group follow-up uses 20 images from each of the 12 race/gender groups, matching the other feature folders. Each source image gets an original plus 19 alpha variants. You may change `GROUPS` below to run other groups. Alpha tiles and overlays are laid out in ascending alpha order.

To keep “Run All” from starting a long GPU sweep unexpectedly, set `RUN_EXPERIMENT = True` only when ready. The sweep uses the existing local model and SAE files and does not download weights.

A previous run with explicitly prompted scarf/no-scarf portraits is archived separately under `race_gender_feature_interventions/feature_138/run_002`; it is not the biased-dataset run.

In [ ]:
from pathlib import Path
import os
import runpy
import subprocess
import sys

# Match the other feature folders: 20 images from every race/gender group.
GROUPS = ["black_man", "black_woman", "white_man", "white_woman", "east_asian_man", "east_asian_woman", "middle_eastern_man", "middle_eastern_woman", "indian_man", "indian_woman", "hispanic_man", "hispanic_woman"]
ALPHAS = [-100, -80, -60, -50, -40, -30, -20, -10, -5, 0, 5, 10, 20, 30, 40, 50, 60, 80, 100]
RUN_EXPERIMENT = False
RENDER_VISUALS = False

def ancestors(path):
    path = Path(path).resolve()
    return [path, *path.parents]

roots = [p for p in ancestors(Path.cwd()) if (p / "data/datasets/biased/images").is_dir()]
if not roots and Path("/n/fs/goose/data/datasets/biased/images").is_dir():
    roots = [Path("/n/fs/goose")]
if not roots:
    raise FileNotFoundError("Could not find data/datasets/biased/images. Set SAE_PROJECT_ROOT to the workspace root.")
PROJECT_ROOT = Path(os.environ.get("SAE_PROJECT_ROOT", roots[0])).expanduser().resolve()
OUTPUT_DIR = PROJECT_ROOT / "sae-activation-tuning/tuning_activations/feature_138/run_biased_002"

script_candidates = []
for parent in ancestors(Path.cwd()):
    script_candidates.extend([
        parent / "sae-activation-tuning/tuning_activations/feature_138/run_biased_dataset_sweep.py",
        parent / "sae-demo-notebooks/sae-activation/activation_tuning/run_biased_dataset_sweep.py",
        parent / "sae-activation-tuning/race_gender_feature_interventions/feature_138/run_biased_dataset_sweep.py",
        parent / "cos398/tuning_activations/feature_138/run_biased_dataset_sweep.py",
    ])
RUNNER = next((p for p in script_candidates if p.is_file()), None)
RENDERER = next((p.with_name("render_signed_visuals.py") for p in script_candidates
                 if p.is_file() and p.with_name("render_signed_visuals.py").is_file()), None)
if RUNNER is None or RENDERER is None:
    raise FileNotFoundError("Could not locate the sweep and overlay scripts alongside this notebook.")

print("Project root:", PROJECT_ROOT)
print("Groups:", GROUPS)
print("Alphas:", ALPHAS)
print("Output:", OUTPUT_DIR)
print("Runner:", RUNNER)
print("Renderer:", RENDERER)

## Run the biased dataset intervention

The all-group follow-up processes the first 20 seeds in each of the 12 groups at 19 alpha values. The original focused 100-image run remains available in `run_biased_001`. Results are written to a new directory; existing results are never overwritten.

In [ ]:
if RUN_EXPERIMENT:
    command = [sys.executable, str(RUNNER), "--groups", *GROUPS, "--output", str(OUTPUT_DIR)]
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print("Set RUN_EXPERIMENT = True above to start the GPU sweep.")

## Render the translucent signed overlays

Zero is transparent. Negative coefficients move from green through blue to purple; positive coefficients move from yellow through orange to red. Each map tile is scaled individually for visibility, so compare alpha magnitudes using the graph and metrics. The original and alpha-zero tiles show the pre-intervention activation map to mark the feature location that was suppressed; the actual post-reencoded activation at alpha zero is zero.

In [ ]:
if RENDER_VISUALS:
    if not (OUTPUT_DIR / "activation_maps.npz").is_file():
        raise FileNotFoundError(f"No completed sweep at {OUTPUT_DIR}")
    os.environ["SAE_FEATURE_RUN"] = str(OUTPUT_DIR)
    runpy.run_path(str(RENDERER))["main"]()
else:
    print("Set RENDER_VISUALS = True after the sweep completes to rebuild charts and overlays.")

In [ ]:
from IPython.display import Image as DisplayImage, display

for group in GROUPS:
    graph = OUTPUT_DIR / group / "activation_graphs" / f"{group}.png"
    sheet = OUTPUT_DIR / group / "seed_sheets" / f"{group}_seed000.jpg"
    overlay = OUTPUT_DIR / group / "signed_coefficient_overlays" / f"{group}_seed0.jpg"
    for path in (graph, sheet, overlay):
        if path.exists():
            print(path.name)
            display(DisplayImage(filename=str(path)))